In [1]:
!pip install boto3 pandas pyarrow fsspec s3fs

INFO: pip is looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of aiobotocore to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/

In [3]:
# import io
# import json
import boto3
import pandas as pd
# from botocore.client import Config

s3 = boto3.client(
    "s3",
    endpoint_url="http://seaweedfs:8333",
    aws_access_key_id="admin",
    aws_secret_access_key="secret"
)

In [5]:
s3.create_bucket(Bucket="bronze2")
s3.create_bucket(Bucket="silver2")
s3.create_bucket(Bucket="gold2")

{'ResponseMetadata': {'RequestId': '18DB3609F30EDD2CCF1F8BCB',
  'HostId': '',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'accept-ranges': 'bytes',
   'content-length': '0',
   'location': '/gold2',
   'server': 'SeaweedFS 30GB 4.48',
   'vary': 'Origin',
   'x-amz-request-id': '18DB3609F30EDD2CCF1F8BCB',
   'date': 'Sun, 04 Oct 2026 03:31:33 GMT'},
  'RetryAttempts': 0},
 'Location': '/gold2'}

In [6]:
# Producer to Bronze
s3.upload_file(
    Filename='sales2.csv',
    Bucket='bronze2',
    Key='sales2.csv'
)

In [15]:
df = pd.read_csv('sales2.csv', sep=';')
df

,id,store,product,qty,price
0,1,s1,p1,2,1000
1,2,s2,p2,3,2000
2,3,s3,p3,2,2000
3,4,s2,p2,20,1000


In [16]:
df.dtypes

id         int64
store        str
product      str
qty        int64
price      int64
dtype: object

In [22]:
import io 
from io import BytesIO # dari Ai
from botocore.client import Config

# Schema yang diharapkan
schema = {
    "id": "int64",
    "store": "str",
    "product": "str",
    "qty": "int64",
    "price": "int64"
}

actual_schema = df.dtypes.astype(str).to_dict()
try:
    if actual_schema != schema:
        print("Schema tidak sesuai!\n")
        raise ValueError("Schema Berbeda")
    buffer = BytesIO()
    df.to_parquet(buffer, index=False)
    s3 = boto3.client(
        "s3",
        endpoint_url="http://seaweedfs:8333",
        aws_access_key_id="admin",
        aws_secret_access_key="secret",
        config=Config(
            s3={"addressing_style": "path"},
            request_checksum_calculation="when_required"
        )
    )
    s3.put_object(
        Bucket="bronze",
        Key="sales.parquet",
        Body=buffer.getvalue()
    )
    print("Sesuai schema!")
    print("Data behasil masuk Bronze!")
except ValueError as e:
    print("\nError:", e)

Sesuai schema!
Data behasil masuk Bronze!
